In [ ]:
!pip install --upgrade numpy --target ./python
!pip install --upgrade numexpr --target ./python

In [ ]:
import sys
sys.path.append(r"./python")

import os
import json
from model import *

#根据时间情况修改index和language值
index =  "mtr_to_qsts_demo_0212_claude"

embedding_endpoint_name = "cohere.embed-multilingual-v3"

embedding_type = 'bedrock' if embedding_endpoint_name.find('titan') or embedding_endpoint_name.find('cohere') else 'sagemaker'
embeddings = init_embeddings_bedrock(embedding_endpoint_name)

In [ ]:
import sys
sys.path.append(r"./python")

from tqdm import tqdm
import fitz
from PIL import Image
import numpy as np
import base64
from opensearch_multimodel_dataload import add_multimodel_documents
import re
import io
import time


model_name = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

# model_name = "amazon.nova-pro-v1:0"

llm = init_model_bedrock(model_name)
text_max_length = 300
llm_max_size = 800

def is_json(myjson):
    try:
        json.loads(myjson)
    except ValueError as e:
        return False
    return True

# prompt = """
# You are a document manager at an real estate company and your task is to extract useful information from document.
# <instructions>
# 1.Output the document in markdown format. 
# 2.Extract all the content of the document,don't omit information, don’t make up the content.
# 3.if the document contains table, keep the rows and columns aligned for the table.
# 4.summarize page content to facilitate searching, keep the summary as brief as possible, put the summary in the <summarize></summarize> tag and put it at the bottom of the content.
# 5.No preface, just output the document content directly.
# </instructions>
# """

prompt = """
You are a document analyst at an subway company and your task is to extract useful information from document.
##instructions##
    1.Extract all the content of the document,include the footer and header, don't omit information, don’t make up the content.
    2.if the document contains table, output the table as markdown format,keep the rows and columns aligned for the table. Note that if there is a table within a table, the table inside also needs to be converted to markdown format
    3.No preface, just output the document content directly.
    4.Generate as many question-answer pairs as possible based on the table or list content to facilitate testing of train driving trainers, The answer should strictly refer to the document content and be detailed.The question and answer pairs are placed in <QA><QA> tags after page content, where the question is placed in <question></question> tags and the answer is placed in <answer></answer> tags
    
"""

# prompt = 'Provide art titles for this image'

files_path = '../docs/MTR2/'
# os.mkdir('images/')
files = os.listdir(files_path)
for file in files:
    page_content = ''
    file_path = files_path + file
    print(file_path)
    fname = file.split('/')[-1].split('.')[0]
    print(fname)
    # os.mkdir('images/'+fname)

    doc = fitz.open(file_path)
    
    texts = []
    metadatas = []
    images = []
    paragraph_combine = ''
    last_title = ''
    multi_response_list = []
    image_base64_list = []
    
    for i in tqdm(range(doc.page_count)):
        print('i:',i)
        
        if i < 0:
            continue
        else:
            time.sleep(60)
            page = doc.load_page(i)
            pix = page.get_pixmap(dpi=200)

            image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            print(image.size)
            if image.size[0] > llm_max_size:
                image = image.resize((llm_max_size,int(image.size[1]*llm_max_size / image.size[0])))
                print('resize image size:',image.size)

            buffered = io.BytesIO()
            image.save(buffered, format="JPEG")
            base64_string = base64.b64encode(buffered.getvalue()).decode("utf-8")

            model_kwargs = {'image': base64_string,'image_type':'jpeg','max_tokens':10000}
            llm.model_kwargs = model_kwargs
            response = llm(prompt)
            response = response.replace('July 2022','').replace('Issue/Rev: 1.0','').replace('Printed versions may not be timely updated. The on-line version is the most up-to-date.','').replace('TO-OST-S','').strip()
            
            response_list = response.split('<QA>',1)
            current_page = response_list[0].strip()
            current_page_list = current_page.split('\n')
            print('current_page_list:',current_page_list)

            title_list = current_page_list[0].split(',')
            if len(title_list) == 1:
                title = title_list[0].strip()
            elif len(title_list) == 2:
                title = title_list[1].strip()
            print('current_page_list 0:',current_page_list[0])
            print('title:',title)
            
            level_2_title = ''
            if len(current_page_list) > 2:
                level_2_title = current_page_list[1].split('.')  if len(current_page_list[1].strip()) > 0 else current_page_list[2].split('.')
            print('current_page_list 1:',current_page_list[1])
            print('current_page_list 2:',current_page_list[2])
            print('level_2_title:',level_2_title)
            
            if last_title == '':
                last_title = title
            
            elif title != last_title or len(level_2_title) == 2 or len(multi_response_list) > 10 or title.find('Abbreviations') >= 0 or i == doc.page_count-1:
                print('begin to save the pages')
                for page_response in multi_response_list:
                    content_set = set()
                    content_list = page_response.split('\n')
                    content_list = content_list[1:] #remove the title

                    for content in content_list:
                        if len(content) > text_max_length:
                            sentence_list = content.split('.')
                            for sentence in sentence_list:
                                if len(sentence) > 0:
                                    content_set.add(sentence)
                        elif len(content.strip()) > 0 and content.strip().find('QA') < 0:
                            content_set.add(content.strip())

                    for text in content_set:
                        text = text.strip()
                        print('text:',text)
                        if len(text) > 0:
                            metadata = {}
                            metadata['sentence'] = text[:text_max_length] if len(text) > text_max_length else text
                            metadata['source'] = file.split('/')[-1]
                            texts.append(paragraph_combine)
                            print('image_base64_list len:',len(image_base64_list))
                            images.append(image_base64_list)
                            metadatas.append(metadata)

                    if len(texts) > 0:
                        if embedding_type == 'bedrock':
                            text_embeddings = embeddings.embed_documents([metadata['sentence'] for metadata in metadatas])
                        else:
                            text_embeddings = embeddings.embed_documents([metadata['sentence'] for metadata in metadatas],chunk_size=10)

                        print('texts len:',len(texts))
                        print('metadatas len:',len(metadatas))
                        print('embeddings len:',len(text_embeddings))
                        print('images len:',len(images))
                        print('begin to save in vectore store')

                        add_multimodel_documents(
                            index,
                            texts=texts,
                            embeddings=text_embeddings,
                            metadatas=metadatas,
                            images=images
                        )
                    print('finish save in vectore store:',index)
                    
                    texts = []
                    metadatas = []
                    images = []
                paragraph_combine = ''
                multi_response_list = []
                image_base64_list = []
            
            if response.find('Intentionally Blank') <0:
                multi_response_list.append(response)
                image_base64_list.append(base64_string)
                last_title = title 
                if len(response_list) > 1:
                    qa = '<QA>' + response_list[1].strip()
                    paragraph_combine += ('<page>' + str(i+1) + '</page> <content>' + current_page + '</content>' + qa + '\n')
                else:
                    paragraph_combine += ('<page>' + str(i+1) + '</page> <content>' + current_page + '</content>' + '\n')
                print(paragraph_combine)
            print('****************************')
         

